<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Consolidación final de los productos priorizados

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp, tables

### Parámetros

Define el decil a partir del cual la calidad se considera alta y los cuantiles que delimitan
los productos de mayor tamaño de mercado dentro de cada margen.

In [3]:
decil_alto = 8

cuantil_90 = 0.9
cuantil_80 = 0.8

### Lectura de datos

Carga la cartera con tamaño de mercado, la correspondencia con la clasificación industrial y
las salidas de los criterios calculados en los notebooks anteriores.

In [4]:
df_products = pd.read_parquet(temp / 'df_products_mercado.parquet')
df_ciiu = pd.read_parquet(temp / 'df_ciiu.parquet')
df_ciiu3 = pd.read_parquet(raw / 'ciiu' / 'ciiu3.parquet')

### Criterios de calidad

Convierte cada criterio en un indicador comparable. El primero se escala por el número de
grupos de destino en los que el producto alcanza calidad alta; el tercero es binario; el cuarto
compara el decil de referencia con el umbral de calidad alta, tanto para Colombia como para los
tres países de comparación.

El notebook original repetía el bloque del cuarto criterio una vez por país; aquí se recorre
con la tabla de parámetros.

In [5]:
df_calidad_1 = pd.read_excel(tables / 'calidad' / 'criterio1.xlsx')
df_calidad_1['HS6'] = df_calidad_1['HS6'].astype(str).str.zfill(6)
df_calidad_1['Calidad C1'] = ((df_calidad_1['Grupos'] - 1) / 6)

df_calidad_3 = pd.read_excel(tables / 'calidad' / 'criterio3.xlsx')
df_calidad_3['HS6'] = df_calidad_3['HS6'].astype(str).str.zfill(6)
df_calidad_3['Calidad C3'] = 1


def leer_criterio4(ruta):
    df = pd.read_excel(ruta)
    df['HS6'] = df['HS6'].astype(str).str.zfill(6)
    df['Decile Promedio'] = df['Decile Promedio'].fillna(0)
    df['Calidad C4'] = np.where(df['Decile Promedio'] >= decil_alto, 1, 0)
    return df


df_calidad_4 = leer_criterio4(tables / 'calidad' / 'criterio4.xlsx')
df_calidad_MEX_c4 = leer_criterio4(tables / 'calidad' / 'otros países' / 'mex_criterio4.xlsx')
df_calidad_CHL_c4 = leer_criterio4(tables / 'calidad' / 'otros países' / 'chl_criterio4.xlsx')
df_calidad_PER_c4 = leer_criterio4(tables / 'calidad' / 'otros países' / 'per_criterio4.xlsx')

### Resiliencia, medidas comerciales y transporte

Carga el resto de criterios. Los costos de transporte se normalizan dentro de cada margen entre
cero y uno, y el interno se invierte para que un valor alto signifique ventaja.

In [6]:
df_resiliencia = pd.read_excel(tables / 'resiliencia' / 'df_resiliencia.xlsx')
df_resiliencia['HS6'] = df_resiliencia['HS6'].astype(str).str.zfill(6)

df_resiliencia_nec = pd.read_excel(tables / 'resiliencia' / 'df_resiliencia_nec_declarada.xlsx')
df_resiliencia_nec['HS6'] = df_resiliencia_nec['HS6'].astype(str).str.zfill(6)

df_tariffs = pd.read_excel(tables / 'tariffs' / 'df_tariffs.xlsx')
df_tariffs['HS6'] = df_tariffs['HS6'].astype(str).str.zfill(6)
df_tariffs['Arancel'] = df_tariffs['Ventaja Arancelaria']

df_tariffs_trump = pd.read_excel(tables / 'tariffs' / 'df_tariffs_trump.xlsx')
df_tariffs_trump['HS6'] = df_tariffs_trump['HS6'].astype(str).str.zfill(6)
df_tariffs_trump['Arancel Trump'] = df_tariffs_trump['Ventaja Arancelaria']

df_ntms = pd.read_excel(tables / 'tariffs' / 'df_ntms.xlsx')
df_ntms['HS6'] = df_ntms['HS6'].astype(str).str.zfill(6)
df_ntms['NTMs'] = df_ntms['Ventaja no Arancelaria']

df_trans_inter = pd.read_excel(tables / 'localizacion' / 'loc internos.xlsx')
df_trans_inter['HS6'] = df_trans_inter['HS6'].astype(str).str.zfill(6)
df_trans_inter['Transporte interno'] = ((df_trans_inter['COSTO POR KG'] - df_trans_inter.groupby('Margen')['COSTO POR KG'].transform('min'))/(df_trans_inter.groupby('Margen')['COSTO POR KG'].transform('max')-df_trans_inter.groupby('Margen')['COSTO POR KG'].transform('min'))).round(2)
df_trans_inter['Transporte interno'] = 1 - df_trans_inter['Transporte interno']

df_trans_ext = pd.read_excel(tables / 'localizacion' / 'loc externos.xlsx')
df_trans_ext['HS6'] = df_trans_ext['HS6'].astype(str).str.zfill(6)
df_trans_ext['Transporte externo Aire'] = ((df_trans_ext['Air'] - df_trans_ext.groupby('Margen')['Air'].transform('min'))/(df_trans_ext.groupby('Margen')['Air'].transform('max')-df_trans_ext.groupby('Margen')['Air'].transform('min'))).round(2)
df_trans_ext['Transporte externo Mar'] = ((df_trans_ext['Sea'] - df_trans_ext.groupby('Margen')['Sea'].transform('min'))/(df_trans_ext.groupby('Margen')['Sea'].transform('max')-df_trans_ext.groupby('Margen')['Sea'].transform('min'))).round(2)

### Consolidación de los criterios

Adjunta a cada producto los indicadores de calidad, resiliencia, medidas comerciales,
transporte y clasificación industrial.

In [7]:
df_products = (df_products.merge(
    df_calidad_1[['HS6','Calidad C1']], on='HS6', how='left'
).merge(
    df_calidad_3[['HS6','Calidad C3']], on='HS6', how='left'
).merge(
    df_calidad_4[['HS6','Calidad C4']], on='HS6', how='left'
).merge(
    df_calidad_MEX_c4[['HS6','Calidad C4']].rename(columns={'Calidad C4': 'Calidad C4 MEX'}), on='HS6', how='left'
).merge(
    df_calidad_CHL_c4[['HS6','Calidad C4']].rename(columns={'Calidad C4': 'Calidad C4 CHL'}), on='HS6', how='left', suffixes=('_MEX', '_CHL')
).merge(
    df_calidad_PER_c4[['HS6','Calidad C4']].rename(columns={'Calidad C4': 'Calidad C4 PER'}), on='HS6', how='left', suffixes=('', '_PER')
).fillna(0).merge(
    df_resiliencia[['HS6','Resiliencia']], on='HS6', how='left'
).merge(
    df_resiliencia_nec[['HS6','Resiliencia ND']], on='HS6', how='left'
).merge(
    df_tariffs[['HS6','Arancel']], on='HS6', how='left'
).merge(
    df_tariffs_trump[['HS6','Arancel Trump']], on='HS6', how='left'
).merge(
    df_ntms[['HS6','NTMs']], on='HS6', how='left'
).fillna({'NTMs':0}).merge(
    df_trans_inter[['HS6','Transporte interno']].drop_duplicates(), on='HS6', how='left'
).merge(
    df_trans_ext[['HS6','Transporte externo Aire','Transporte externo Mar']].drop_duplicates(), on='HS6', how='left'
).merge(
    df_ciiu.drop_duplicates(), on='HS6', how='left'
)).drop_duplicates()

### Indicadores agregados y cortes de tamaño de mercado

Promedia los criterios por dimensión y marca los productos que superan los cuantiles noventa y
ochenta del potencial exportador restante dentro de su margen.

In [8]:
df_products['Calidad Total'] = df_products[['Calidad C1', 'Calidad C3']].mean(axis=1)
df_products['Calidad 4 MEX PER CHL'] = df_products[['Calidad C4 MEX', 'Calidad C4 PER', 'Calidad C4 CHL']].mean(axis=1)

df_products['Arancel Total'] = df_products[['Arancel', 'NTMs']].mean(axis=1)
df_products['Arancel Trump Total'] = df_products[['NTMs', 'Arancel Trump']].mean(axis=1)
df_products['Transporte Total'] = df_products[['Transporte interno', 'Transporte externo Aire', 'Transporte externo Mar']].mean(axis=1)

df_products['Cutoff 90'] = df_products.groupby('Margen')['Potencial Exportador Restante'].transform(lambda x: x.quantile(cuantil_90))
df_products['Cutoff 80'] = df_products.groupby('Margen')['Potencial Exportador Restante'].transform(lambda x: x.quantile(cuantil_80))
df_products['Tamaño de Mercado 90'] = np.where(df_products['Potencial Exportador Restante'] >= df_products['Cutoff 90'],1,0)
df_products['Tamaño de Mercado 80'] = np.where(df_products['Potencial Exportador Restante'] >= df_products['Cutoff 80'],1,0)

df_products['CIIU 3digit'] = df_products['CIIU'].str[:3]
df_products = df_products.merge(
    df_ciiu3[['CIIU 3digit', 'Descripción']].rename(columns={'Descripción': 'Industria 3digit'}),
    on='CIIU 3digit',
    how='left'
)

### Exportación del consolidado

Guarda la hoja de productos completos, con su fuente y categoría, y la de productos únicos, que
es la que consume la cadena de estimación de impacto.

In [9]:
df_products = df_products[['HS6','Description','Fuente','Categoría','Margen','Leamer','Type','Potencial Exportador Restante','Cutoff 90', 'Cutoff 80', 'Tamaño de Mercado 90', 'Tamaño de Mercado 80','CIIU','Industria','CIIU 3digit','Industria 3digit','Calidad Total','Calidad C4','Calidad 4 MEX PER CHL','Resiliencia', 'Resiliencia ND','Arancel Total','Arancel Trump Total','Transporte Total']]

df_products_unicos = df_products[['HS6','Description','Margen','Leamer','Type','Potencial Exportador Restante','Cutoff 90', 'Cutoff 80', 'Tamaño de Mercado 90', 'Tamaño de Mercado 80','CIIU','Industria','CIIU 3digit','Industria 3digit','Calidad Total','Calidad C4','Calidad 4 MEX PER CHL','Resiliencia', 'Resiliencia ND','Arancel Total','Arancel Trump Total','Transporte Total']].drop_duplicates()

tables.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(tables / 'Output Final 040725.xlsx') as writer:
    df_products.round(3).to_excel(writer, sheet_name="Productos completos", index=False)
    df_products_unicos.round(3).to_excel(writer, sheet_name="Productos únicos", index=False)

print(f"productos completos: {len(df_products):,}   productos únicos: {len(df_products_unicos):,}")

productos completos: 728   productos únicos: 662
